In [1]:
from langchain_ollama import ChatOllama
model = ChatOllama(model="llama3.2:latest")
print('Model is ready')

Model is ready


In [2]:
from langchain_community.tools import tool
from langchain_community.tools import DuckDuckGoSearchResults, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

@tool
def get_news(query:str) -> str:
    """Helps to get news details for any specific subject"""
    duckDuckGoTool = DuckDuckGoSearchResults()
    return duckDuckGoTool.invoke(query)

@tool
def get_wikipedia_information(query: str) -> str:
    """Gets the information about historical topics"""
    print('get_wikipedia_information called')
    wikipedia_api_wrapper = WikipediaAPIWrapper(
        top_k_search=2,
        doc_content_chars_max=500
    )
    wikipedia_tool = WikipediaQueryRun(api_wrapper=wikipedia_api_wrapper)
    result = wikipedia_tool.run(query)
    print(result)
    return f"Result {result}"
get_news
get_wikipedia_information

C:\Users\bg108836\AppData\Local\Temp\ipykernel_14748\1956556730.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import tool


StructuredTool(name='get_wikipedia_information', description='Gets the information about historical topics', args_schema=<class 'langchain_core.utils.pydantic.get_wikipedia_information'>, func=<function get_wikipedia_information at 0x0000023D74C38B80>)

In [10]:
from langchain.agents import create_agent

ALL_TOOLS = [get_news, get_wikipedia_information]

agent = create_agent(
    model=model,
    tools=ALL_TOOLS,
    system_prompt="You are a helpful assistant. You can provide the news information with the help of get_news tool and to get information about history please call get_wikipedia_information tool"
)

print(agent.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	model(model)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> model;
	model -.-> __end__;
	model -.-> tools;
	tools -.-> model;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [15]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "What are the headlines of recent BRICS summit?"}
    ]
})

response

{'messages': [HumanMessage(content='What are the headlines of recent BRICS summit?', additional_kwargs={}, response_metadata={}, id='ee58cd2f-77dd-47d5-9d09-b3ac6785cc26'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-10-24T17:36:14.4775179Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3300007000, 'load_duration': 54329500, 'prompt_eval_count': 238, 'prompt_eval_duration': 206058000, 'eval_count': 17, 'eval_duration': 2886584000, 'logprobs': None, 'model_name': 'llama3.2:latest', 'model_provider': 'ollama'}, id='lc_run--01a16efb-ff9b-778a-b13f-d163feab0eb6-0', tool_calls=[{'name': 'get_news', 'args': {'query': 'BRICS summit recent headlines'}, 'id': '85f61c78-3cc7-4bda-899f-50b183b61ed3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 238, 'output_tokens': 17, 'total_tokens': 255}),
  ToolMessage(content="snippet: Follow the latest on BRICS, including coverage of the BRICS sum

In [18]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "What the phases in the history of Artificial Intelligence?"}
    ]
})

response

get_wikipedia_information called


SSLError: HTTPSConnectionPool(host='en.wikipedia.org', port=443): Max retries exceeded with url: /w/api.php?list=search&srprop=&srlimit=3&limit=3&srsearch=Artificial+Intelligence+history&format=json&action=query (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)')))

In [12]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is the temperature of city of London now?"}
    ]
})

response

{'messages': [HumanMessage(content='What is the temperature of city of London now?', additional_kwargs={}, response_metadata={}, id='4adb2481-14d4-4cf7-95ca-cf1edb55d3c2'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-10-24T17:33:30.4958071Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2986355200, 'load_duration': 21953500, 'prompt_eval_count': 238, 'prompt_eval_duration': 842756000, 'eval_count': 15, 'eval_duration': 2112132999, 'logprobs': None, 'model_name': 'llama3.2:latest', 'model_provider': 'ollama'}, id='lc_run--01a16ef9-8053-76b1-8bca-508fe389586d-0', tool_calls=[{'name': 'get_news', 'args': {'query': 'London temperature now'}, 'id': '922b547e-8f6c-43c4-960b-5d92150952cb', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 238, 'output_tokens': 15, 'total_tokens': 253}),
  ToolMessage(content='snippet: 1 day ago · London weather today, hour by hour: temperature, chance of

In [ ]:
from dotenv import load_dotenv, find_dotenv
import os
env_path = find_dotenv()
load_dotenv(env_path)

tavilyApiKey = os.getenv("TAVILY_API_KEY")
print('Tavily API Key is ready')

In [ ]:
from tavily import TavilyClient
from langchain_community.tools import TavilySearchResults
tavily_tool = TavilySearchResults(max_results=5)
result = tavily_tool.invoke("What are the headlines of recent BRICS summit?")
result